In [ ]:
# =====================================

# FAKE NEWS DETECTION SYSTEM

# IMPROVED VERSION

# =====================================

import pandas as pd
import pickle
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

# =====================================

# LOAD DATASETS

# =====================================

fake = pd.read_csv("/content/Fake.csv")
true = pd.read_csv("/content/True.csv")

# Labels

fake["label"] = 0
true["label"] = 1

# Merge

df = pd.concat([fake, true], ignore_index=True)

# Create Content Column

df["content"] = (
df["title"].fillna("") + " " +
df["text"].fillna("")
)

# Remove Duplicates

df.drop_duplicates(subset=["content"], inplace=True)

# Shuffle Dataset

df = df.sample(frac=1, random_state=42)

# Keep Required Columns

df = df[["content", "label"]]

print("Dataset Shape:", df.shape)

# =====================================

# TEXT CLEANING

# =====================================

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["content"] = df["content"].apply(clean_text)

# =====================================

# FEATURES AND TARGET

# =====================================

X = df["content"]
y = df["label"]

# =====================================

# TRAIN TEST SPLIT

# =====================================

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42,
stratify=y
)

# =====================================

# TF-IDF VECTORIZATION

# =====================================

vectorizer = TfidfVectorizer(
stop_words="english",
lowercase=True,
max_df=0.8,
min_df=2,
ngram_range=(1,2),
max_features=50000
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

# =====================================

# MODEL TRAINING

# =====================================

model = LogisticRegression(
max_iter=1000,
random_state=42
)

model.fit(X_train, y_train)

# =====================================

# MODEL EVALUATION

# =====================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy Score:")
print(round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =====================================

# PREDICTION FUNCTION

# =====================================

def predict_news(news_text):
    news_text = clean_text(news_text)
    transformed_text = vectorizer.transform([news_text])
    prediction = model.predict(transformed_text)
    if prediction[0] == 1:
        return "REAL NEWS"
    else:
        return "FAKE NEWS"

# =====================================

# USER INPUT

# =====================================

print("\n====================================")
print("FAKE NEWS DETECTION SYSTEM")
print("====================================")

while True:
    user_news = input("\nEnter News Article (or type EXIT): ")

    if user_news.upper() == "EXIT":
        break

    if len(user_news.split()) < 10:
        print("Please enter a complete news article.")
        continue

    result = predict_news(user_news)

    print("\nPrediction:", result)

# =====================================

# SAVE MODEL

# =====================================

with open("fake_news_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("\nModel Saved Successfully!")

print("Created Files:")
print("1. fake_news_model.pkl")
print("2. vectorizer.pkl")

Dataset Shape: (39105, 2)

Accuracy Score:
98.66 %

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3582
           1       0.98      0.99      0.99      4239

    accuracy                           0.99      7821
   macro avg       0.99      0.99      0.99      7821
weighted avg       0.99      0.99      0.99      7821


Confusion Matrix:
[[3505   77]
 [  28 4211]]

FAKE NEWS DETECTION SYSTEM

Enter News Article (or type EXIT): WASHINGTON (Reuters) - The United States government announced new economic measures on Tuesday to support small businesses and improve employment opportunities. Officials stated that the policy would be implemented gradually over the coming months. Economists believe the move could strengthen economic growth and increase investments across multiple sectors.

Prediction: REAL NEWS
